# mv-PIKN · Core model smoke test
**Stage 3: verify the model computation before full training.**

Run Sections **1–7** in order. A CPU works; a GPU is optional. The default uses the already checked `outer_chb01 / inner_01` development split. This notebook embeds the loader, so the previous notebook need not be open. It reads your existing preprocessing run.

This is an explicitly specified **core-model prototype**, not a reproduction of the full manuscript. The manuscript leaves fusion widths and several losses undefined. This step uses raw/spectral encoders, learned concatenation fusion, a three-class head, and a rank-8 one-step latent predictor. It omits Wilson–Cowan regularization, reconstruction and stability penalties pending an explicit defensible formulation. No biological interpretation or stability guarantee is claimed.

Three optimizer steps on a tiny class-stratified training sample test computation only. Validation is a short chronological inference check, not a performance estimate. No thresholds, alarms, AUROC or event sensitivity are computed. No outer test samples are loaded.

## 1 · Dependencies and Drive

In [ ]:
%pip -q install numpy pandas h5py
import importlib.util
if importlib.util.find_spec('torch') is None:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch'])
from google.colab import drive
drive.mount('/content/gdrive')

## 2 · Fold-aware data loader
Uses the same loader and strict causal-history availability as the checked data-loading notebook. Metadata scans can take several minutes over Drive.

In [ ]:
"""Fold-scoped, causal CHB-MIT loading. No model fitting or alarm scoring."""
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import h5py

class FoldData:
    def __init__(self, run, outer='outer_chb01', scope='inner_01'):
        self.run = Path(run)
        self.cfg = json.loads((self.run/'config.json').read_text())
        self.inventory = json.loads((self.run/'locked_inventory.json').read_text())
        self.events = json.loads((self.run/'locked_events.json').read_text())
        folds = json.loads((self.run/'locked_folds.json').read_text())
        self.fold = next(f for f in folds if f['id'] == outer)
        self.outer, self.scope = outer, scope
        if scope == 'outer_refit':
            self.roles = {'train': self.fold['development_subjects'], 'test': self.fold['test_subjects']}
        else:
            inner = next(f for f in self.fold['inner_folds'] if f['id'] == scope)
            self.roles = {'train': inner['train_subjects'], 'validation': inner['validation_subjects']}
        if set(self.roles['train']) & set(self.fold['test_subjects']):
            raise ValueError('Held-out subject appears in training.')
        if len(set(sum(self.roles.values(), []))) != len(sum(self.roles.values(), [])):
            raise ValueError('Roles overlap or contain duplicate subjects.')
        status = json.loads((self.run/'status.json').read_text())
        if status.get('status') not in ['review_required','preprocessing_complete']:
            raise ValueError('Preprocessing is incomplete.')
        blocking = [x for x in status.get('cohort_issues', [])
                    if x.get('reason') != 'no_valid_preictal_input_after_QC']
        if blocking:
            raise ValueError(f'Unresolved preprocessing issues: {blocking}')
        if status.get('cohort_issues'):
            audit = json.loads((self.run/'training_label_audit_status_v2_2.json').read_text())
            if audit.get('revision') != '2.2':
                raise ValueError('Run preprocessing Section 7B first.')
        self.scaler_path = self.run/'folds'/outer/scope/'train_scaler.npz'
        with np.load(self.scaler_path, allow_pickle=False) as z:
            if set(z['training_subjects'].tolist()) != set(self.roles['train']):
                raise ValueError('Scaler training subjects do not match the fold.')
            if set(z['test_subjects'].tolist()) != set(self.fold['test_subjects']):
                raise ValueError('Scaler outer test subjects do not match.')
            self.scaler = {k:z[k].copy() for k in ['raw_mean','raw_std','feature_mean','feature_std']}
        C = len(self.cfg['channels'])
        for k,shape in [('raw_mean',(C,)),('raw_std',(C,)),('feature_mean',(C,5)),('feature_std',(C,5))]:
            v = self.scaler[k]
            if v.shape != shape or not np.isfinite(v).all() or ('std' in k and np.any(v<=0)):
                raise ValueError(f'Invalid scaler: {k}')
        ratio = self.cfg['fusion_history_s']/self.cfg['window_s']
        if ratio != int(ratio):
            raise ValueError('Fusion history must contain whole windows.')
        self.history = int(ratio)
        self.subject_index = {s:i for i,s in enumerate(sorted({r['subject'] for r in self.inventory}))}
        self._metadata = {}

    def records(self, role):
        if role not in self.roles:
            raise ValueError(f'{role} is unavailable in {self.scope}; allowed: {list(self.roles)}')
        rows = [r for r in self.inventory if r['subject'] in self.roles[role]
                and r['signal_ok'] and not r['excluded_case'] and r['split']=='pool']
        return sorted(rows, key=lambda r:(r['subject'],r['case'],r['start_s'],r['relative_path']))

    def path(self, row):
        return self.run/'processed'/row['case']/'pool'/(Path(row['relative_path']).stem+'.h5')

    def metadata(self, row):
        p = self.path(row)
        if p not in self._metadata:
            with h5py.File(p,'r') as h:
                if str(h.attrs['case']) != row['case'] or json.loads(h.attrs['channels_json']) != self.cfg['channels']:
                    raise ValueError(f'Unexpected case or channels: {p}')
                if json.loads(h.attrs['config_json']) != self.cfg:
                    raise ValueError(f'Unexpected processing configuration: {p}')
                keys = ['label','valid_signal','window_start_s','decision_s','pair_with_previous']
                m = {k:h[k][:] for k in keys}
            n = len(m['label']); w = self.cfg['window_s']
            if any(len(v)!=n for v in m.values()) or not np.isin(m['label'],[-1,0,1,2]).all():
                raise ValueError(f'Invalid metadata: {p}')
            if n and (not np.allclose(m['decision_s']-m['window_start_s'],w) or
                      np.any(np.diff(m['decision_s'])<=0)):
                raise ValueError(f'Invalid timeline: {p}')
            # History ends before current window starts. Reset on every EDF boundary,
            # missing interval, or invalid signal; never concatenate across files.
            ready = np.zeros(n,dtype=bool)
            streak = 0
            for i in range(n):
                if i and not np.isclose(m['window_start_s'][i],m['decision_s'][i-1],rtol=0,atol=1e-6):
                    streak = 0
                if m['valid_signal'][i]:
                    ready[i] = streak >= self.history
                    streak += 1
                else:
                    streak = 0
            m['prediction_ready'] = ready
            self._metadata[p] = m
        return self._metadata[p]

    def index(self, role, mode='supervised'):
        if mode not in ['supervised','pairs','replay']:
            raise ValueError(mode)
        if mode=='pairs' and role!='train':
            raise ValueError('Future-window pairs are restricted to training.')
        items=[]
        for row in self.records(role):
            m=self.metadata(row); n=len(m['label'])
            if mode=='replay': keep=np.ones(n,dtype=bool)
            else:
                keep=m['prediction_ready'] & (m['label']>=0)
                if mode=='pairs':
                    next_ok=np.zeros(n,dtype=bool)
                    if n>1:
                        next_ok[:-1]=m['valid_signal'][1:] & m['pair_with_previous'][1:]
                    keep &= next_ok
            items.extend((row,int(i)) for i in np.flatnonzero(keep))
        return items

    def read(self, row, i, mode='supervised'):
        if mode not in ['supervised','pairs','replay']: raise ValueError(mode)
        m=self.metadata(row); ready=bool(m['prediction_ready'][i])
        if mode!='replay' and (not ready or m['label'][i]<0):
            raise ValueError('Supervised sample lacks labels or causal history.')
        C=len(self.cfg['channels']); L=int(self.cfg['fs']*self.cfg['window_s']); H=self.history
        inputs={'raw_t':np.zeros((C,L),np.float32), 'spec_t':np.zeros((C,5),np.float32),
                'raw_past':np.zeros((H,C,L),np.float32), 'spec_past':np.zeros((H,C,5),np.float32)}
        def normal(raw,spec):
            s=self.scaler
            x=((raw-s['raw_mean'][:,None])/s['raw_std'][:,None]).astype(np.float32)
            f=((spec-s['feature_mean'])/s['feature_std']).astype(np.float32)
            if not np.isfinite(x).all() or not np.isfinite(f).all():
                raise ValueError('Nonfinite normalized input')
            return x,f
        auxiliary={}
        with h5py.File(self.path(row),'r') as h:
            # Unavailable replay rows retain timestamps but carry zero placeholders.
            # The model must run ONLY when prediction_ready is true.
            if ready:
                inputs['raw_t'],inputs['spec_t']=normal(h['raw_uv'][i],h['features'][i])
                inputs['raw_past'],inputs['spec_past']=normal(h['raw_uv'][i-H:i],h['features'][i-H:i])
            if mode=='pairs':
                if row['subject'] not in self.roles['train']:
                    raise ValueError('Future-window pairs are restricted to training subjects.')
                if i+1>=len(m['label']) or not m['valid_signal'][i+1] or not m['pair_with_previous'][i+1]:
                    raise ValueError('Invalid adjacent pair.')
                auxiliary['raw_tdt'],auxiliary['spec_tdt']=normal(h['raw_uv'][i+1],h['features'][i+1])
        return dict(inputs=inputs, auxiliary=auxiliary,
                    target=dict(label=int(m['label'][i])),
                    metadata=dict(subject=row['subject'],case=row['case'],patient_idx=self.subject_index[row['subject']],
                                  file=str(self.path(row)),index=i,start_s=float(m['window_start_s'][i]),
                                  decision_s=float(m['decision_s'][i]),valid_signal=bool(m['valid_signal'][i]),
                                  prediction_ready=ready, reset_at_file_start=i==0))

    def iterate(self, role, mode='replay', shuffle=False, seed=20261028):
        if shuffle and (role!='train' or mode=='replay'):
            raise ValueError('Only supervised training may be shuffled.')
        items=self.index(role,mode)
        if shuffle:
            rng=np.random.default_rng(seed); rng.shuffle(items)
        for row,i in items: yield self.read(row,i,mode)

    def audit(self, role):
        rows=[];targets=[]
        for row in self.records(role):
            m=self.metadata(row)
            rows.append(dict(subject=row['subject'],file=row['relative_path'],windows=len(m['label']),
                             valid_signal=int(m['valid_signal'].sum()),prediction_ready=int(m['prediction_ready'].sum()),
                             **{f'ready_label_{k}':int((m['prediction_ready']&(m['label']==k)).sum()) for k in [-1,0,1,2]}))
        for e in self.events:
            if not e['eligible_target'] or e['split']!='pool' or e['subject'] not in self.roles[role]:continue
            a=e['onset_s']-self.cfg['sph_s']-self.cfg['sop_s'];b=e['onset_s']-self.cfg['sph_s']
            n=0
            for row in self.records(role):
                if row['case']!=e['case']:continue
                m=self.metadata(row)
                # Decision times, independent of supervised labels.
                n+=int((m['prediction_ready']&(m['decision_s']>=a)&(m['decision_s']<=b)).sum())
            targets.append(dict(event_id=e['event_id'],subject=e['subject'],retain_in_evaluation=True,
                                available_forecast_decisions=n,has_forecast_opportunity=n>0))
        return pd.DataFrame(rows),pd.DataFrame(targets)


## 3 · Explicit model and loss definitions
- Raw branch: depthwise Conv1d(C→C, kernel 7, padding 3), GELU, pointwise Conv1d(C→64), GELU; concatenate global mean/max, then Linear(128→32).
- Spectral branch: flatten C×5; Linear(C×5→64), LayerNorm, GELU, Linear(64→32).
- Fusion: concatenate both latents; Linear(64→32), LayerNorm, GELU. Classifier: Linear(32→3).
- Dynamics: K = I + UVᵀ, U,V each 32×8; predicts the fused latent one four-second step ahead. Both factors start with small random values to avoid zero-factor gradient deadlock.
- Loss: unweighted three-class cross-entropy + 0.5 × MSE(Kzₜ, stop_gradient(zₜ₊₁)). This is a smoke-test weight, not a tuned scientific result.

The operator is an **auxiliary training regularizer**: classification reads the current fused latent, not Kz. The future observation is a detached training target; it never enters the classifier. The trainable encoders remain susceptible to representational collapse, so latent variation is reported diagnostically; this small test cannot establish absence of collapse.

Temporal fusion is separate from latent fusion: average the top eight preictal probabilities among 74 past windows plus the current window (75 total). The existing loader requires 75 valid preceding windows, conservatively delaying availability by one extra window. Preserve that checked policy here. This score is neither an alarm nor a calibrated risk.

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

class CoreMVModel(nn.Module):
    """Explicit smoke-test foundation, not the complete physics-informed model."""
    def __init__(self, channels=18, latent=32, rank=8):
        super().__init__()
        self.raw_conv=nn.Sequential(nn.Conv1d(channels,channels,7,padding=3,groups=channels),nn.GELU(),
                                    nn.Conv1d(channels,64,1),nn.GELU())
        self.raw_project=nn.Linear(128,latent)
        self.spectral=nn.Sequential(nn.Flatten(1),nn.Linear(channels*5,64),nn.LayerNorm(64),nn.GELU(),nn.Linear(64,latent))
        self.fusion=nn.Sequential(nn.Linear(latent*2,latent),nn.LayerNorm(latent),nn.GELU())
        self.classifier=nn.Linear(latent,3)
        self.U=nn.Parameter(torch.randn(latent,rank)*0.01)
        self.V=nn.Parameter(torch.randn(latent,rank)*0.01)

    def encode(self,raw,spec):
        h=self.raw_conv(raw)
        zr=self.raw_project(torch.cat([h.mean(-1),h.amax(-1)],dim=-1))
        zs=self.spectral(spec)
        return self.fusion(torch.cat([zr,zs],dim=-1))

    def propagate(self,z):
        # Row-vector convention for K=I+UV^T: z_next = z K^T.
        return z+(z@self.V)@self.U.T

    def forward(self,raw,spec):
        z=self.encode(raw,spec)
        return self.classifier(z),z

    def operator_matrix(self):
        return torch.eye(self.U.shape[0],device=self.U.device)+self.U@self.V.T


def tensor_batch(samples,device,paired=False):
    def stack(group,key):
        return torch.as_tensor(np.stack([s[group][key] for s in samples]),dtype=torch.float32,device=device)
    out={'raw':stack('inputs','raw_t'),'spec':stack('inputs','spec_t'),
         'labels':torch.tensor([s['target']['label'] for s in samples],dtype=torch.long,device=device)}
    if paired:
        out.update(next_raw=stack('auxiliary','raw_tdt'),next_spec=stack('auxiliary','spec_tdt'))
    return out


def core_loss(model,batch,dynamics_weight=0.5):
    logits,z=model(batch['raw'],batch['spec'])
    ce=F.cross_entropy(logits,batch['labels'])
    with torch.no_grad():
        target=model.encode(batch['next_raw'],batch['next_spec'])
    dynamics=F.mse_loss(model.propagate(z),target)
    return ce+dynamics_weight*dynamics,{'classification':ce,'dynamics':dynamics},z


def check_gradients(model):
    report={}
    for name,p in model.named_parameters():
        if p.grad is None or not torch.isfinite(p.grad).all():
            raise RuntimeError(f'Missing/nonfinite gradient: {name}')
        report[name]=float(p.grad.norm().detach().cpu())
    if not any(x>0 for x in report.values()): raise RuntimeError('All gradients are zero')
    return report


@torch.no_grad()
def causal_probabilities(model, sample, device, top_k=8, encode_chunk=16):
    if not sample['metadata']['prediction_ready']:
        return None
    inputs=sample['inputs']
    # Manuscript convention: last 74 preceding windows + current = 75.
    # Loader requires 75 preceding valid windows, so its availability rule is
    # conservatively one window stricter. Preserve it; never silently relax it.
    H=len(inputs['raw_past'])
    if H<top_k or H<1: raise ValueError('Insufficient history for Top-K score')
    raw=np.concatenate([inputs['raw_past'][1:],inputs['raw_t'][None]],axis=0)
    spec=np.concatenate([inputs['spec_past'][1:],inputs['spec_t'][None]],axis=0)
    probabilities=[]
    was_training=model.training;model.eval()
    try:
        for start in range(0,len(raw),encode_chunk):
            logits,_=model(torch.as_tensor(raw[start:start+encode_chunk],device=device),
                           torch.as_tensor(spec[start:start+encode_chunk],device=device))
            probabilities.append(logits.softmax(-1).cpu())
    finally:
        model.train(was_training)
    p=torch.cat(probabilities).numpy()
    return dict(p_interictal=float(p[-1,0]),p_preictal=float(p[-1,1]),p_ictal=float(p[-1,2]),
                top8_preictal=float(np.sort(p[:,1])[-top_k:].mean()))


def smoke_sample_indices(data,per_class=2,seed=20261028):
    # Deliberately tiny class-stratified engineering test, not a training sampler.
    candidates=data.index('train','pairs');rng=np.random.default_rng(seed)
    picked=[]
    for label in [0,1,2]:
        eligible=[x for x in candidates if data.metadata(x[0])['label'][x[1]]==label]
        if not eligible: raise RuntimeError(f'No class {label} dynamics pairs in this training split')
        chosen=rng.choice(len(eligible),size=min(per_class,len(eligible)),replace=False)
        picked.extend(eligible[int(i)] for i in chosen)
    return picked


## 4 · Initialize the development fit

In [ ]:
import random
SEED=20261028
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type=='cpu': torch.set_num_threads(min(4,torch.get_num_threads()))
RUN=Path('/content/gdrive/MyDrive/mvPIKN_preprocessing/run_20261028T165011846018Z')
OUTER='outer_chb01'; SCOPE='inner_01'
data=FoldData(RUN,OUTER,SCOPE)
if SCOPE=='outer_refit': raise RuntimeError('This smoke test must use an inner development split.')
model=CoreMVModel(channels=len(data.cfg['channels'])).to(DEVICE)
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
print('Device:',DEVICE,'PyTorch:',torch.__version__)
print('Training subjects:',data.roles['train'])
print('Validation subjects:',data.roles['validation'])
print('Excluded outer test:',data.fold['test_subjects'])
parameter_counts={name:sum(p.numel() for p in module.parameters()) for name,module in model.named_children()}
parameter_counts['dynamics_U_V']=model.U.numel()+model.V.numel()
parameter_counts['total']=sum(p.numel() for p in model.parameters())
print('Parameter counts:',parameter_counts)

## 5 · Three training steps and gradient checks
Uses up to two adjacent-window examples per class, all from training subjects. Samples are deliberately stratified to exercise all logits; this is not the eventual training sampling strategy. The same tiny batch is reused three times. History arrays are not passed into the window classifier; they are used separately during causal score fusion.

In [ ]:
chosen=smoke_sample_indices(data,per_class=2,seed=SEED)
samples=[data.read(row,i,mode='pairs') for row,i in chosen]
batch=tensor_batch(samples,DEVICE,paired=True)
print('Batch shapes:',{k:tuple(v.shape) for k,v in batch.items()})
print('Training labels:',batch['labels'].tolist())
training_log=[]
initial={k:p.detach().clone() for k,p in model.named_parameters()}
model.train()
for step in range(3):
    optimizer.zero_grad(set_to_none=True)
    loss,parts,z=core_loss(model,batch)
    if not torch.isfinite(loss): raise RuntimeError('Nonfinite loss')
    loss.backward()
    gradient_norms=check_gradients(model)
    for branch in ['raw_conv','raw_project','spectral','fusion','classifier']:
        if not any(v>0 for k,v in gradient_norms.items() if k.startswith(branch+'.')):
            raise RuntimeError('No gradient in '+branch)
    if gradient_norms['U']==0 or gradient_norms['V']==0:
        raise RuntimeError('Dynamics factor has zero gradient')
    total_grad_norm=torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
    optimizer.step()
    if not all(torch.isfinite(p).all() for p in model.parameters()):
        raise RuntimeError('Nonfinite parameters after update')
    training_log.append(dict(step=step+1,total=float(loss.detach().cpu()),
                             **{k:float(v.detach().cpu()) for k,v in parts.items()},
                             preclip_gradient_norm=float(total_grad_norm.detach().cpu()),
                             latent_mean_std=float(z.detach().std(dim=0,unbiased=False).mean().cpu())))
changed=[k for k,p in model.named_parameters() if not torch.equal(initial[k],p.detach())]
if not changed: raise RuntimeError('Optimizer did not update any parameters')
display(pd.DataFrame(training_log))
print('Parameters updated:',len(changed),'/',len(initial))
print('Training smoke test passed. These losses do not measure generalization.')

## 6 · Short chronological validation replay
Keeps unavailable rows and their timestamps; predicts only when the loader marks history ready. Stops at eight ready predictions or 1,000 stored windows, whichever comes first. Predictions use neither labels nor future-window targets. This is a short preview, not full validation coverage. Top-K defaults are fixed for this engineering test and are not selected using these outputs.

In [ ]:
from itertools import islice
model.eval(); validation_preview=[];ready_count=0
with torch.no_grad():
    for sample in islice(data.iterate('validation',mode='replay'),1000):
        meta=sample['metadata']
        prediction=causal_probabilities(model,sample,DEVICE)
        row={k:meta[k] for k in ['subject','case','file','index','decision_s','valid_signal','prediction_ready']}
        row.update(prediction if prediction is not None else
                   dict(p_interictal=np.nan,p_preictal=np.nan,p_ictal=np.nan,top8_preictal=np.nan))
        validation_preview.append(row)
        if prediction is not None:
            ready_count+=1
            if abs(sum(prediction[k] for k in ['p_interictal','p_preictal','p_ictal'])-1)>1e-5:
                raise RuntimeError('Probabilities do not sum to one')
            if not all(np.isfinite(v) and 0<=v<=1 for v in prediction.values()):
                raise RuntimeError('Invalid probability')
        if ready_count>=8: break
preview=pd.DataFrame(validation_preview)
display(preview.tail(12))
print('Chronological rows checked:',len(preview),'ready predictions:',ready_count)
if ready_count==0:
    raise RuntimeError('No ready validation windows in the first 1,000 rows. Inspect coverage; do not relax masks automatically.')
print('Validation inference smoke test passed; alarm scoring has not run.')

## 7 · Save smoke-test diagnostics
Creates a fresh timestamped folder under `model_smoke_v1`. The checkpoint is for debugging only; initialize a fresh model for proper training. No preprocessing outputs are replaced. Send the JSON summary and three-step loss table.

In [ ]:
from datetime import datetime, timezone
OUTPUT=RUN/'model_smoke_v1'/OUTER/SCOPE/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT.mkdir(parents=True,exist_ok=False)
pd.DataFrame(training_log).to_csv(OUTPUT/'training_smoke_losses.csv',index=False)
preview.to_csv(OUTPUT/'validation_smoke_preview.csv',index=False)
model_config=dict(channels=len(data.cfg['channels']),latent=32,rank=8)
summary=dict(stage='core_model_smoke_v1',status='smoke_passed_full_training_and_scoring_not_run',
             outer=OUTER,scope=SCOPE,seed=SEED,device=str(DEVICE),torch_version=str(torch.__version__),
             roles=data.roles,model_config=model_config,parameter_counts=parameter_counts,
             training_samples=len(chosen),training_steps=3,validation_rows=len(preview),
             validation_ready_predictions=ready_count,classification_classes=[0,1,2],
             dynamics_weight=0.5,physics_regularizer_implemented=False,
             operator_role='auxiliary detached-next-fused-latent prediction loss',
             temporal_fusion='top 8 of 74 previous plus current window; existing 75-preceding availability mask retained',
             scaler_sha256=hashlib.sha256(data.scaler_path.read_bytes()).hexdigest(),
             folds_sha256=hashlib.sha256((RUN/'locked_folds.json').read_bytes()).hexdigest(),
             training_sample_ids=[dict(subject=r['subject'],file=r['relative_path'],index=i) for r,i in chosen])
(OUTPUT/'smoke_summary.json').write_text(json.dumps(summary,indent=2))
torch.save(dict(model_state_dict=model.state_dict(),model_config=model_config,
                smoke_only=True,seed=SEED),OUTPUT/'debug_only_checkpoint.pt')
print(json.dumps(summary,indent=2))
print('Saved:',OUTPUT)

## What passing means
Inputs, normalization, forward/backward computation, optimizer updates and bounded chronological inference work for this development fit. It does not establish accuracy, stable dynamics, physiological meaning or clinical utility. The operator is not constrained to have eigenvalues inside the unit circle.

Before full experiments, specify the missing physics formulation and ablations, training schedule, sampling/weights, and event/alarm scoring. Select model settings only within development folds. Never use this tiny stratified batch to estimate class prevalence, thresholds, calibration or performance.

Local synthetic tests cover finite gradients, detached future targets, operator orientation, probability normalization, unavailable rows and invariance of predictions to labels/future-target metadata. Your Colab run is the first integration check on the actual EEG.